# CODe — FULL Qwen2.5-VL-3B reproduction on RTX PRO 6000

Notebook này chạy **end-to-end offline**:

```text
verify full assets
    ↓
install local wheels
    ↓
full zero-shot cls test
    ↓
LoRA SFT full cls_train
    ↓
3 epochs
    ↓
best validation-loss checkpoint
    ↓
full fine-tuned cls test
    ↓
save predictions + metrics + adapter
```

## Mức độ bám theo tác giả

Các setting trực tiếp từ paper/released config:

- Qwen2.5-VL-3B-Instruct
- LoRA
- target `all` → ở đây ánh xạ PEFT `all-linear`
- cutoff 4096
- max samples 30000
- per-device batch size 2
- gradient accumulation 8
- learning rate 5e-4
- cosine LR
- warmup ratio 0.1
- BF16
- **3 epochs theo paper** (released YAML hiện để 1 epoch)

Các điểm **không thể exact-match vì nguồn public không công bố đủ**:

- paper nói giữ best validation checkpoint nhưng không công bố validation split; notebook tạo 5% validation từ `cls_train.json`, seed 42, và **không dùng test để chọn model**;
- released repo không cung cấp prompt file mà `evaluate.py` tham chiếu; notebook dùng chính `question/messages` trong released benchmark;
- repo không công bố code tính Table-3 metrics; notebook báo exact-set accuracy + multilabel micro/macro F1 + single-label subset metrics;
- paper dùng 2×A800 cho 3B; notebook này dùng 1×RTX PRO 6000.

Đây là **released-code-faithful reproduction trên hardware hiện có**, không gọi là bit-exact reproduction.

## v2 robust fixes

Bản này đã audit và sửa các điểm dễ fail trước khi chạy nhiều giờ:

- không còn bulk-install toàn bộ wheelhouse lên Kaggle;
- bảo vệ `torch/torchvision/Pillow/numpy/pandas`;
- subprocess import preflight + tự repair Pillow nếu gặp lỗi `_Ink`;
- hỗ trợ asset dạng folder hoặc `.tar`;
- train schema hỗ trợ `messages`, `conversations`, hoặc flat `q/a`;
- model load single-GPU sạch, không dùng `device_map="auto"` khi sẽ train;
- inference batch size 2 giống released `evaluate.py`;
- bỏ phụ thuộc scikit-learn để tránh ABI mismatch;
- forward/backward memory probe trước training;
- tự fallback batch `2×8 -> 1×16` nếu OOM, giữ effective batch size 16;
- gradient checkpointing an toàn hơn với PEFT;
- checkpoint resume sort theo số thay vì chuỗi;
- lưu environment/version + mọi deviation vào output.


## v3 Kaggle-safe environment logic

- Không còn dừng chỉ vì `PIL` hoặc library đã xuất hiện trong `sys.modules`.
- Kaggle/Jupyter có thể preload PIL cho phần display; đó là bình thường.
- Chạy fresh-subprocess import preflight **trước khi pip install**.
- Nếu môi trường Kaggle đã tương thích, notebook bỏ qua pip install hoàn toàn.
- Chỉ cài selective ML wheels khi thật sự cần.
- Không đụng Pillow trừ khi preflight xác nhận Pillow đang lỗi.


In [1]:
from pathlib import Path
import os, sys, subprocess, json, ast, re, time, random, shutil, inspect, math
import importlib
import importlib.metadata as importlib_metadata

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
OUT = WORK / "code_full_reproduction"
OUT.mkdir(parents=True, exist_ok=True)

SEED = 42
RUN_ZERO_SHOT = True
RUN_TRAIN = True
RUN_FINETUNED_EVAL = True

EVAL_BATCH_SIZE = 2
EVAL_MAX_NEW_TOKENS = 4096

random.seed(SEED)

print("Python:", sys.version)
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    raise RuntimeError(
        "Không tìm thấy nvidia-smi. Hãy bật RTX PRO 6000 trước khi chạy notebook này."
    )

# Kaggle/Jupyter có thể preload PIL cho display nội bộ.
# Việc này KHÔNG có nghĩa kernel bẩn.
preloaded = sorted({
    name.split(".")[0]
    for name in sys.modules
    if name == "PIL" or name.startswith("PIL.")
    or name == "transformers" or name.startswith("transformers.")
    or name == "peft" or name.startswith("peft.")
    or name == "accelerate" or name.startswith("accelerate.")
})
print("Preloaded relevant modules:", preloaded if preloaded else "none")
print("Compatibility sẽ được kiểm tra bằng một fresh subprocess ở bước tiếp theo.")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]
Tue Sep 29 10:33:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.159.04             Driver Version: 580.159.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX PRO 6000 Blac...    Off |   00000000:05:00.0 Off |                    0 |
| N/A   30C    P0             47W /  600W |       0MiB /  97887MiB |      0%      Default |
|                                         |                        |             

## 1. Auto-discover full offline assets

In [2]:
def discover_direct_asset(root):
    roots = []
    for p in root.rglob("cls_train.json"):
        candidate = p.parent.parent
        if (
            (candidate / "data" / "cls_test.csv").exists()
            and (candidate / "Qwen2.5-VL-3B-Instruct" / "config.json").exists()
        ):
            roots.append(candidate)
    return roots

asset_roots = discover_direct_asset(INPUT)

# Optional support for the tar workflow if Kaggle Dataset creation required archives.
if not asset_roots:
    runtime_tars = list(INPUT.rglob("code_runtime_data.tar"))
    model_tars = list(INPUT.rglob("qwen25_vl_3b.tar"))

    if runtime_tars and model_tars:
        import tarfile

        extracted = WORK / "code_assets_extracted"
        marker = extracted / ".extract_ok"

        if not marker.exists():
            if extracted.exists():
                shutil.rmtree(extracted)
            extracted.mkdir(parents=True, exist_ok=True)

            print("Extracting runtime tar:", runtime_tars[0])
            with tarfile.open(runtime_tars[0], "r") as tar:
                tar.extractall(extracted)

            print("Extracting model tar:", model_tars[0])
            with tarfile.open(model_tars[0], "r") as tar:
                tar.extractall(extracted)

            marker.write_text("ok", encoding="utf-8")

        if (
            (extracted / "data" / "cls_train.json").exists()
            and (extracted / "data" / "cls_test.csv").exists()
            and (extracted / "Qwen2.5-VL-3B-Instruct" / "config.json").exists()
        ):
            asset_roots = [extracted]

if not asset_roots:
    raise FileNotFoundError(
        "Không thấy FULL asset bundle. Cần attach output FULL từ notebook 00, "
        "hoặc attach dataset chứa code_runtime_data.tar + qwen25_vl_3b.tar."
    )

ASSET = asset_roots[0]
DATA = ASSET / "data"
MODEL_DIR = ASSET / "Qwen2.5-VL-3B-Instruct"
WHEEL_DIR = ASSET / "wheelhouse"
TRAIN_JSON = DATA / "cls_train.json"
TEST_CSV = DATA / "cls_test.csv"

print("ASSET:", ASSET)
print("MODEL:", MODEL_DIR)
print("TRAIN:", TRAIN_JSON)
print("TEST :", TEST_CSV)
print("WHEELS:", WHEEL_DIR)

assert TRAIN_JSON.exists(), TRAIN_JSON
assert TEST_CSV.exists(), TEST_CSV
assert (MODEL_DIR / "config.json").exists(), MODEL_DIR
assert WHEEL_DIR.exists(), WHEEL_DIR

ASSET: /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets
MODEL: /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/Qwen2.5-VL-3B-Instruct
TRAIN: /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/data/cls_train.json
TEST : /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/data/cls_test.csv
WHEELS: /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/wheelhouse


## 2. Offline wheel install

In [3]:
# Strategy:
# 1) Test môi trường Kaggle hiện tại bằng FRESH Python subprocess.
# 2) Nếu đã tương thích -> KHÔNG pip install gì cả.
# 3) Chỉ nếu fail mới cài selective ML wheels.
# Như vậy tránh làm hỏng Pillow/torchvision/numpy stack của Kaggle.

wheels = sorted(WHEEL_DIR.glob("*.whl"))
print("Wheel files available:", len(wheels))

def wheel_dist_name(path):
    return path.name.split("-", 1)[0].replace("_", "-").lower()

wheel_by_dist = {}
for w in wheels:
    wheel_by_dist.setdefault(wheel_dist_name(w), []).append(w)

def run_import_preflight():
    script = r"""
from PIL import Image, ImageDraw, ImageFont
import importlib.metadata as md
import torch
import torchvision
import transformers
import peft
import accelerate
from qwen_vl_utils import process_vision_info
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from peft import LoraConfig, get_peft_model
print("IMPORT_PREFLIGHT_OK")
print("Pillow", md.version("Pillow"))
print("torch", torch.__version__)
print("torchvision", torchvision.__version__)
print("transformers", transformers.__version__)
print("peft", peft.__version__)
print("accelerate", accelerate.__version__)
"""
    return subprocess.run(
        [sys.executable, "-c", script],
        text=True,
        capture_output=True,
    )

def show_preflight(label, result):
    print()
    print(f"=== {label} STDOUT ===")
    print(result.stdout)
    if result.stderr.strip():
        print(f"=== {label} STDERR ===")
        print(result.stderr)

preflight = run_import_preflight()
show_preflight("SYSTEM PREFLIGHT", preflight)

installed_anything = False
repaired_pillow = False

if preflight.returncode == 0:
    print("✅ Kaggle system environment đã tương thích.")
    print("Bỏ qua toàn bộ pip install từ wheelhouse.")
else:
    print("System env chưa đủ. Thử cài selective ML wheels...")

    wanted = [
        "transformers",
        "peft",
        "accelerate",
        "qwen-vl-utils",
        "huggingface-hub",
        "safetensors",
        "tokenizers",
        "av",
    ]

    selected = []
    for dist in wanted:
        choices = wheel_by_dist.get(dist, [])
        if choices:
            selected.append(sorted(choices)[-1])
        else:
            print("Không có offline wheel, giữ bản Kaggle/dependency hiện có:", dist)

    print()
    print("Selected offline ML wheels:")
    for w in selected:
        print(" -", w.name)

    if selected:
        subprocess.run(
            [
                sys.executable, "-m", "pip", "install",
                "--no-index", "--no-deps", "--upgrade",
                *[str(w) for w in selected],
            ],
            check=True,
        )
        installed_anything = True
        importlib.invalidate_caches()

    preflight = run_import_preflight()
    show_preflight("AFTER ML WHEELS", preflight)

    # Known Pillow mismatch recovery.
    if preflight.returncode != 0 and (
        "_Ink" in preflight.stderr
        or "PIL" in preflight.stderr
        or "Pillow" in preflight.stderr
    ):
        pillow_choices = wheel_by_dist.get("pillow", [])

        if pillow_choices:
            pillow_wheel = sorted(pillow_choices)[-1]
            print("Repairing Pillow from:", pillow_wheel.name)

            subprocess.run(
                [
                    sys.executable, "-m", "pip", "install",
                    "--no-index", "--no-deps",
                    "--force-reinstall",
                    str(pillow_wheel),
                ],
                check=True,
            )
            installed_anything = True
            repaired_pillow = True
            importlib.invalidate_caches()

            preflight = run_import_preflight()
            show_preflight("AFTER PILLOW REPAIR", preflight)

if preflight.returncode != 0:
    raise RuntimeError(
        "Python environment vẫn chưa tương thích. "
        "Xem PRE-FLIGHT STDERR ngay phía trên. "
        "Notebook dừng trước khi load model để không tốn GPU."
    )

print("✅ IMPORT_PREFLIGHT_OK")

# Nếu selective install đã thay các ML package mà Kaggle preload trước đó,
# purge riêng các module ML đó để lần import tiếp theo đọc version mới.
# KHÔNG purge PIL/torch/torchvision.
if installed_anything:
    purge_roots = {"transformers", "peft", "accelerate", "qwen_vl_utils"}
    purge_names = [
        name for name in list(sys.modules)
        if name.split(".")[0] in purge_roots
    ]
    for name in purge_names:
        sys.modules.pop(name, None)
    importlib.invalidate_caches()
    print("Purged preloaded ML modules after selective install:", len(purge_names))

# Nếu chính Pillow vừa bị repair mà PIL đã ở RAM,
# cần restart kernel một lần để tránh trộn old-memory/new-files.
# Trường hợp bình thường fresh Kaggle + system preflight pass sẽ không vào nhánh này.
if repaired_pillow and any(
    name == "PIL" or name.startswith("PIL.")
    for name in sys.modules
):
    raise RuntimeError(
        "Pillow vừa được repair trên disk nhưng PIL đã được preload trong kernel. "
        "Hãy Restart Session MỘT LẦN rồi chạy lại. "
        "Lần sau system preflight sẽ pass và notebook sẽ không cài Pillow nữa."
    )

pip_check = subprocess.run(
    [sys.executable, "-m", "pip", "check"],
    text=True,
    capture_output=True,
)
print("pip check return code:", pip_check.returncode)
if pip_check.stdout.strip():
    print(pip_check.stdout[:5000])

Wheel files available: 38

=== SYSTEM PREFLIGHT STDOUT ===

=== SYSTEM PREFLIGHT STDERR ===
Traceback (most recent call last):
  File "<string>", line 9, in <module>
ModuleNotFoundError: No module named 'qwen_vl_utils'

System env chưa đủ. Thử cài selective ML wheels...

Selected offline ML wheels:
 - transformers-4.57.6-py3-none-any.whl
 - peft-0.21.0-py3-none-any.whl
 - accelerate-1.15.0-py3-none-any.whl
 - qwen_vl_utils-0.0.14-py3-none-any.whl
 - huggingface_hub-0.36.2-py3-none-any.whl
 - safetensors-0.8.0-cp310-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
 - tokenizers-0.22.2-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
 - av-18.1.0-cp311-abi3-manylinux_2_28_x86_64.whl
Processing /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/wheelhouse/transformers-4.57.6-py3-none-any.whl
Processing /kaggle/input/datasets/ngoanhkhoa143/code-full-offline-assets-2/code_full_offline_assets/wheelhouse/peft-0.21.0-py3-none-any.whl
Processing 

In [4]:
import numpy as np
import pandas as pd
import torch
import torchvision
import transformers
import peft
import accelerate
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info
from peft import LoraConfig, get_peft_model

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("Pillow:", importlib_metadata.version("Pillow"))
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("accelerate:", accelerate.__version__)
print("CUDA:", torch.cuda.is_available())
print("BF16:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

if not torch.cuda.is_available():
    raise RuntimeError("Notebook FULL reproduction cần GPU.")

np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

# Save environment versions for reproducibility.
env_info = {
    "python": sys.version,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pillow": importlib_metadata.version("Pillow"),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "accelerate": accelerate.__version__,
    "gpu": torch.cuda.get_device_name(0),
}
(OUT / "environment.json").write_text(
    json.dumps(env_info, indent=2),
    encoding="utf-8",
)

numpy: 2.0.2
pandas: 2.3.3
Pillow: 11.3.0
torch: 2.10.0+cu128
torchvision: 0.25.0+cu128
transformers: 4.57.6
peft: 0.21.0
accelerate: 1.15.0
CUDA: True
BF16: True


322

## 3. Load full train/test và helper functions

In [5]:
train_data = json.loads(TRAIN_JSON.read_text(encoding="utf-8"))
test_df = pd.read_csv(TEST_CSV).fillna("")

print("Train:", len(train_data))
print("Test :", len(test_df))
print("Test columns:", list(test_df.columns))

if not isinstance(train_data, list) or len(train_data) == 0:
    raise RuntimeError("cls_train.json không phải list hoặc đang rỗng.")

def parse_images(x):
    if isinstance(x, (list, tuple)):
        return [str(v) for v in x]
    s = str(x).strip()
    if not s:
        return []
    try:
        obj = ast.literal_eval(s)
        if isinstance(obj, (list, tuple)):
            return [str(v) for v in obj]
    except Exception:
        pass
    return [v.strip() for v in s.split(",") if v.strip()]

def local_path(rel):
    rel = str(rel).replace("\\", "/").lstrip("/")
    p = ASSET / rel
    if not p.exists():
        raise FileNotFoundError(f"Missing packaged image: {rel}")
    return p

IMAGE_TOKEN_LINE = re.compile(r"(?im)^\s*image-\d+\s*:\s*<image>\s*$")

def clean_image_placeholders(text):
    s = str(text)
    s = IMAGE_TOKEN_LINE.sub("", s)
    s = s.replace("<image>", "")
    s = re.sub(r"\n{3,}", "\n\n", s)
    return s.strip()

def content_to_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for part in content:
            if isinstance(part, str):
                parts.append(part)
            elif isinstance(part, dict):
                if str(part.get("type", "")).lower() == "text":
                    parts.append(str(part.get("text", "")))
                elif "text" in part:
                    parts.append(str(part.get("text", "")))
        return "\n".join(x for x in parts if x)
    if content is None:
        return ""
    return str(content)

def extract_messages(item):
    user = ""
    assistant = ""

    # OpenAI/messages format
    msgs = item.get("messages")
    if isinstance(msgs, list):
        for m in msgs:
            role = str(m.get("role", m.get("from", ""))).lower()
            content = content_to_text(m.get("content", m.get("value", "")))
            if role in {"user", "human"} and not user:
                user = content
            elif role in {"assistant", "gpt"} and not assistant:
                assistant = content

    # ShareGPT format
    conv = item.get("conversations")
    if isinstance(conv, list):
        for m in conv:
            role = str(m.get("from", m.get("role", ""))).lower()
            content = content_to_text(m.get("value", m.get("content", "")))
            if role in {"user", "human"} and not user:
                user = content
            elif role in {"assistant", "gpt"} and not assistant:
                assistant = content

    # Flat fallback
    if not user:
        user = item.get("question", item.get("q", item.get("prompt", "")))
    if not assistant:
        assistant = item.get("answer", item.get("a", item.get("label", "")))

    user = clean_image_placeholders(content_to_text(user))
    assistant = content_to_text(assistant).strip()

    image_field = (
        item.get("images")
        if "images" in item
        else item.get("image", item.get("photos", item.get("photographs", [])))
    )
    images = [local_path(r) for r in parse_images(image_field)]

    if not user:
        raise ValueError("Train item không đọc được user/question text.")
    if not assistant:
        raise ValueError("Train item không đọc được assistant/answer text.")
    if not images:
        raise ValueError("Train item không có image path hợp lệ.")

    return user, assistant, images

# test schema aliases
lower = {str(c).lower(): c for c in test_df.columns}
TEST_IMAGES_COL = next((lower[x] for x in ["images","image","photos","photographs"] if x in lower), None)
TEST_Q_COL = next((lower[x] for x in ["question","q","prompt","user"] if x in lower), None)
TEST_A_COL = next((lower[x] for x in ["answer","a","label","assistant"] if x in lower), None)
TEST_ID_COL = next((lower[x] for x in ["id","checkup_id","patient_id"] if x in lower), None)

if TEST_IMAGES_COL is None or TEST_Q_COL is None or TEST_A_COL is None:
    raise RuntimeError("cls_test.csv thiếu images/question/answer aliases.")

print("Resolved test:", {
    "id": TEST_ID_COL,
    "images": TEST_IMAGES_COL,
    "q": TEST_Q_COL,
    "a": TEST_A_COL,
})

# Fail fast before loading the model: validate schemas and image paths.
for i, item in enumerate(train_data[:20]):
    user, assistant, paths = extract_messages(item)
    assert all(p.exists() for p in paths), i

for i, (_, row) in enumerate(test_df.head(20).iterrows()):
    refs = parse_images(row[TEST_IMAGES_COL])
    if not refs:
        raise RuntimeError(f"Test row {i} không có ảnh.")
    for ref in refs:
        local_path(ref)

print("✅ Data/schema preflight passed on first 20 train + 20 test samples.")

Train: 14696
Test : 1793
Test columns: ['id', 'images', 'question', 'answer']
Resolved test: {'id': 'id', 'images': 'images', 'q': 'question', 'a': 'answer'}
✅ Data/schema preflight passed on first 20 train + 20 test samples.


## 4. Model + processor

In [6]:
dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

processor = AutoProcessor.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
)
processor.tokenizer.padding_side = "left"

# Avoid device_map='auto' for a model that will later be trained with Trainer.
# The 3B BF16 model comfortably fits on RTX PRO 6000.
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    str(MODEL_DIR),
    torch_dtype=dtype,
    low_cpu_mem_usage=True,
    local_files_only=True,
)
model.to("cuda")
model.eval()

print("dtype:", dtype)
print("device:", next(model.parameters()).device)
print("Allocated GB:", round(torch.cuda.memory_allocated() / 1024**3, 2))
print("Reserved  GB:", round(torch.cuda.memory_reserved() / 1024**3, 2))

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

dtype: torch.bfloat16
device: cuda:0
Allocated GB: 7.1
Reserved  GB: 7.33


## 5. Full zero-shot evaluation

In [7]:
CLASSES = [
    "Caries",
    "Gingivitis",
    "Malocclusion",
    "Pulpitis",
    "Tooth Loss",
    "Tooth Structure Loss",
]

def parse_class_set(text):
    s = str(text).lower()
    found = []
    for cls in sorted(CLASSES, key=len, reverse=True):
        if cls.lower() in s:
            found.append(cls)
    return sorted(set(found))

def safe_div(a, b):
    return float(a / b) if b else 0.0

def compute_metrics(rows):
    df = pd.DataFrame(rows).copy()
    df["true_set"] = df["true"].map(parse_class_set)
    df["pred_set"] = df["pred"].map(parse_class_set)

    exact = np.mean([a == b for a, b in zip(df["true_set"], df["pred_set"])])

    y_true = np.array(
        [[int(c in s) for c in CLASSES] for s in df["true_set"]],
        dtype=np.int32,
    )
    y_pred = np.array(
        [[int(c in s) for c in CLASSES] for s in df["pred_set"]],
        dtype=np.int32,
    )

    tp = int(((y_true == 1) & (y_pred == 1)).sum())
    fp = int(((y_true == 0) & (y_pred == 1)).sum())
    fn = int(((y_true == 1) & (y_pred == 0)).sum())

    micro_precision = safe_div(tp, tp + fp)
    micro_recall = safe_div(tp, tp + fn)
    micro_f1 = safe_div(
        2 * micro_precision * micro_recall,
        micro_precision + micro_recall,
    )

    class_f1 = []
    for j in range(len(CLASSES)):
        yt = y_true[:, j]
        yp = y_pred[:, j]
        ctp = int(((yt == 1) & (yp == 1)).sum())
        cfp = int(((yt == 0) & (yp == 1)).sum())
        cfn = int(((yt == 1) & (yp == 0)).sum())
        p = safe_div(ctp, ctp + cfp)
        r = safe_div(ctp, ctp + cfn)
        class_f1.append(safe_div(2 * p * r, p + r))

    metrics = {
        "n": len(df),
        "exact_set_accuracy": float(exact),
        "micro_f1": float(micro_f1),
        "macro_f1": float(np.mean(class_f1)),
        "micro_precision": float(micro_precision),
        "micro_recall": float(micro_recall),
        "parsed_prediction_rate": float(np.mean([len(x) > 0 for x in df["pred_set"]])),
    }

    one = df[
        (df["true_set"].map(len) == 1)
        & (df["pred_set"].map(len) == 1)
    ].copy()
    if len(one):
        metrics["single_label_parsed_n"] = int(len(one))
        metrics["single_label_accuracy"] = float(np.mean([
            a[0] == b[0]
            for a, b in zip(one["true_set"], one["pred_set"])
        ]))

    return metrics, df

def build_eval_messages(row):
    refs = parse_images(row[TEST_IMAGES_COL])
    paths = [local_path(r) for r in refs]
    q = clean_image_placeholders(row[TEST_Q_COL])

    content = [{"type": "image", "image": str(p)} for p in paths]
    content.append({"type": "text", "text": q})
    return [{"role": "user", "content": content}]

@torch.inference_mode()
def generate_batch(rows, model_obj, max_new_tokens=EVAL_MAX_NEW_TOKENS):
    batch_messages = [build_eval_messages(row) for row in rows]
    texts = [
        processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        for messages in batch_messages
    ]

    # Matches the batching pattern in the released evaluate.py.
    image_inputs, video_inputs = process_vision_info(batch_messages)

    inputs = processor(
        text=texts,
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to("cuda")

    out = model_obj.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
    )
    trimmed = [
        out_ids[len(in_ids):]
        for in_ids, out_ids in zip(inputs.input_ids, out)
    ]
    return processor.batch_decode(
        trimmed,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=False,
    )

def run_full_generation(tag, model_obj):
    rows_out = []
    out_csv = OUT / f"{tag}_predictions.csv"
    total = len(test_df)

    for start in range(0, total, EVAL_BATCH_SIZE):
        batch_df = test_df.iloc[start:start + EVAL_BATCH_SIZE]
        batch_rows = [row for _, row in batch_df.iterrows()]
        t0 = time.time()

        try:
            preds = generate_batch(batch_rows, model_obj)
            errors = [""] * len(batch_rows)
        except Exception as batch_error:
            # Robust fallback: isolate a bad/large case instead of losing the full run.
            print("Batch failed; retrying samples individually:", repr(batch_error))
            preds = []
            errors = []
            for row in batch_rows:
                try:
                    pred = generate_batch([row], model_obj)[0]
                    preds.append(pred)
                    errors.append("")
                except Exception as e:
                    preds.append("")
                    errors.append(repr(e))
                torch.cuda.empty_cache()

        elapsed = time.time() - t0

        for offset, (row, pred, err) in enumerate(zip(batch_rows, preds, errors)):
            pos = start + offset + 1
            rows_out.append({
                "id": row[TEST_ID_COL] if TEST_ID_COL else pos,
                "true": row[TEST_A_COL],
                "pred": pred,
                "error": err,
                "seconds_batch": elapsed,
                "n_images": len(parse_images(row[TEST_IMAGES_COL])),
            })

        pd.DataFrame(rows_out).to_csv(
            out_csv,
            index=False,
            encoding="utf-8-sig",
        )

        done = min(start + len(batch_rows), total)
        if done % 10 == 0 or done == total:
            print(
                f"{tag}: {done}/{total} | "
                f"errors={sum(bool(x['error']) for x in rows_out)} | "
                f"last_batch={elapsed:.1f}s"
            )
        torch.cuda.empty_cache()

    metrics, parsed = compute_metrics(rows_out)
    (OUT / f"{tag}_metrics.json").write_text(
        json.dumps(metrics, indent=2),
        encoding="utf-8",
    )
    parsed.to_csv(
        OUT / f"{tag}_parsed.csv",
        index=False,
        encoding="utf-8-sig",
    )
    print(json.dumps(metrics, indent=2))
    return rows_out, metrics

if RUN_ZERO_SHOT:
    zero_rows, zero_metrics = run_full_generation("zero_shot_base3b", model)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


zero_shot_base3b: 10/1793 | errors=0 | last_batch=8.2s
zero_shot_base3b: 20/1793 | errors=0 | last_batch=5.3s
zero_shot_base3b: 30/1793 | errors=0 | last_batch=6.3s
zero_shot_base3b: 40/1793 | errors=0 | last_batch=8.0s
zero_shot_base3b: 50/1793 | errors=0 | last_batch=6.9s
zero_shot_base3b: 60/1793 | errors=0 | last_batch=4.5s
zero_shot_base3b: 70/1793 | errors=0 | last_batch=2.9s
zero_shot_base3b: 80/1793 | errors=0 | last_batch=6.8s
zero_shot_base3b: 90/1793 | errors=0 | last_batch=1.6s
zero_shot_base3b: 100/1793 | errors=0 | last_batch=4.4s
zero_shot_base3b: 110/1793 | errors=0 | last_batch=5.8s
zero_shot_base3b: 120/1793 | errors=0 | last_batch=3.7s
zero_shot_base3b: 130/1793 | errors=0 | last_batch=6.4s
zero_shot_base3b: 140/1793 | errors=0 | last_batch=6.3s
zero_shot_base3b: 150/1793 | errors=0 | last_batch=7.0s
zero_shot_base3b: 160/1793 | errors=0 | last_batch=5.9s
zero_shot_base3b: 170/1793 | errors=0 | last_batch=3.8s
zero_shot_base3b: 180/1793 | errors=0 | last_batch=2.2s
z

## 6. Full LoRA SFT — released-config-faithful

Important: training uses a 5% validation split sampled **only from released training JSON**, seed 42.

The official paper says the best validation checkpoint was kept, but does not expose the exact validation split. This notebook never uses test labels to select a checkpoint.

In [8]:
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence
from transformers import Trainer, TrainingArguments

class RawCODeDataset(Dataset):
    def __init__(self, items):
        self.items = items
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        return self.items[idx]

# The paper says the best validation checkpoint was retained but the exact
# validation split is not public. Use a deterministic 95/5 split from TRAIN ONLY.
idx = np.arange(len(train_data))
rng = np.random.default_rng(SEED)
rng.shuffle(idx)

n_val = max(1, int(round(0.05 * len(idx))))
val_idx = set(idx[:n_val].tolist())

train_items = [x for i, x in enumerate(train_data) if i not in val_idx]
val_items = [x for i, x in enumerate(train_data) if i in val_idx]

# Released YAML max_samples=30000.
train_items = train_items[:30000]

print("Released train total:", len(train_data))
print("SFT train:", len(train_items))
print("Validation:", len(val_items))

Released train total: 14696
SFT train: 13961
Validation: 735


In [9]:
processor.tokenizer.padding_side = "right"

class QwenVLTrainCollator:
    def __init__(self, processor, max_length=4096):
        self.processor = processor
        self.max_length = max_length

    def _encode_one(self, item):
        user, assistant, image_paths = extract_messages(item)

        user_content = [
            {"type": "image", "image": str(p)}
            for p in image_paths
        ]
        user_content.append({"type": "text", "text": user})

        prompt_messages = [{"role": "user", "content": user_content}]
        full_messages = prompt_messages + [
            {
                "role": "assistant",
                "content": [{"type": "text", "text": assistant}],
            }
        ]

        prompt_text = self.processor.apply_chat_template(
            prompt_messages,
            tokenize=False,
            add_generation_prompt=True,
        )
        full_text = self.processor.apply_chat_template(
            full_messages,
            tokenize=False,
            add_generation_prompt=False,
        )

        image_inputs, video_inputs = process_vision_info(full_messages)

        full = self.processor(
            text=[full_text],
            images=image_inputs,
            videos=video_inputs,
            padding=False,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        prompt = self.processor(
            text=[prompt_text],
            images=image_inputs,
            videos=video_inputs,
            padding=False,
            truncation=True,
            max_length=self.max_length,
            return_tensors="pt",
        )

        input_ids = full["input_ids"][0]
        attention_mask = full["attention_mask"][0]
        labels = input_ids.clone()

        prompt_len = min(prompt["input_ids"].shape[1], labels.shape[0])
        labels[:prompt_len] = -100

        if torch.all(labels == -100):
            raise RuntimeError(
                "Một training sample bị truncate hết assistant answer ở cutoff_len=4096."
            )

        result = {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": labels,
        }

        for key in [
            "pixel_values",
            "image_grid_thw",
            "pixel_values_videos",
            "video_grid_thw",
        ]:
            if key in full:
                result[key] = full[key]

        return result

    def __call__(self, features):
        encs = [self._encode_one(x) for x in features]
        pad_id = self.processor.tokenizer.pad_token_id
        if pad_id is None:
            pad_id = self.processor.tokenizer.eos_token_id

        batch = {
            "input_ids": pad_sequence(
                [x["input_ids"] for x in encs],
                batch_first=True,
                padding_value=pad_id,
            ),
            "attention_mask": pad_sequence(
                [x["attention_mask"] for x in encs],
                batch_first=True,
                padding_value=0,
            ),
            "labels": pad_sequence(
                [x["labels"] for x in encs],
                batch_first=True,
                padding_value=-100,
            ),
        }

        for key in [
            "pixel_values",
            "image_grid_thw",
            "pixel_values_videos",
            "video_grid_thw",
        ]:
            vals = [x[key] for x in encs if key in x]
            if vals:
                batch[key] = torch.cat(vals, dim=0)

        return batch

collator = QwenVLTrainCollator(processor, max_length=4096)

# Encode-only smoke tests before LoRA/training.
for n in [1, 2]:
    smoke_batch = collator(train_items[:n])
    print(
        f"Collator batch={n}:",
        {k: tuple(v.shape) for k, v in smoke_batch.items()},
    )
    del smoke_batch

torch.cuda.empty_cache()
print("✅ Collator preflight passed.")

Collator batch=1: {'input_ids': (1, 1890), 'attention_mask': (1, 1890), 'labels': (1, 1890), 'pixel_values': (7232, 1176), 'image_grid_thw': (10, 3)}
Collator batch=2: {'input_ids': (2, 1890), 'attention_mask': (2, 1890), 'labels': (2, 1890), 'pixel_values': (10304, 1176), 'image_grid_thw': (14, 3)}
✅ Collator preflight passed.


In [10]:
if RUN_TRAIN:
    model.train()
    model.config.use_cache = False

    # Non-reentrant checkpointing is more robust with PEFT/frozen base weights.
    if hasattr(model, "gradient_checkpointing_enable"):
        try:
            model.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs={"use_reentrant": False}
            )
        except TypeError:
            model.gradient_checkpointing_enable()

    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        lora_dropout=0.0,
        bias="none",
        target_modules="all-linear",
        task_type="CAUSAL_LM",
    )

    try:
        model = get_peft_model(model, lora_config)
    except Exception as e:
        print("PEFT all-linear failed:", repr(e))
        print("Falling back to all discovered Linear module leaf names.")

        linear_names = set()
        for name, module in model.named_modules():
            if isinstance(module, torch.nn.Linear) and not name.endswith("lm_head"):
                linear_names.add(name.split(".")[-1])

        if not linear_names:
            raise RuntimeError("Không discover được Linear modules cho LoRA.") from e

        print("Fallback LoRA targets:", sorted(linear_names))
        lora_config = LoraConfig(
            r=8,
            lora_alpha=16,
            lora_dropout=0.0,
            bias="none",
            target_modules=sorted(linear_names),
            task_type="CAUSAL_LM",
        )
        model = get_peft_model(model, lora_config)

    model.print_trainable_parameters()

    # Probe author batch size=2 before starting a multi-hour run.
    # If RTX PRO 6000 still OOMs for a worst-ish first batch, preserve effective
    # batch size by switching 2x8 -> 1x16 and record the deviation.
    TRAIN_BATCH_SIZE = 2
    GRAD_ACC_STEPS = 8
    batch_fallback_reason = None

    try:
        probe = collator(train_items[:2])
        probe = {
            k: v.to("cuda")
            for k, v in probe.items()
        }
        probe_out = model(**probe)
        probe_out.loss.backward()
        model.zero_grad(set_to_none=True)
        del probe_out, probe
        torch.cuda.empty_cache()
        print("✅ Forward/backward probe succeeded at batch_size=2.")
    except torch.cuda.OutOfMemoryError as e:
        batch_fallback_reason = repr(e)
        print("OOM at batch_size=2; fallback to batch_size=1, grad_acc=16.")
        TRAIN_BATCH_SIZE = 1
        GRAD_ACC_STEPS = 16
        model.zero_grad(set_to_none=True)
        torch.cuda.empty_cache()

        # Verify fallback before the real training job.
        probe = collator(train_items[:1])
        probe = {k: v.to("cuda") for k, v in probe.items()}
        probe_out = model(**probe)
        probe_out.loss.backward()
        model.zero_grad(set_to_none=True)
        del probe_out, probe
        torch.cuda.empty_cache()
        print("✅ Forward/backward fallback probe succeeded.")

    TRAIN_OUT = OUT / "lora_3b_3epochs"

    # Released config uses save_steps=1000. If this release is too small to reach
    # 1000 total optimizer steps, evaluate/save once per approximate epoch instead.
    steps_per_epoch = math.ceil(
        len(train_items) / (TRAIN_BATCH_SIZE * GRAD_ACC_STEPS)
    )
    total_steps_est = steps_per_epoch * 3
    SAVE_EVAL_STEPS = 1000 if total_steps_est >= 1000 else max(1, steps_per_epoch)

    print("TRAIN_BATCH_SIZE:", TRAIN_BATCH_SIZE)
    print("GRAD_ACC_STEPS:", GRAD_ACC_STEPS)
    print("Estimated optimizer steps:", total_steps_est)
    print("save/eval steps:", SAVE_EVAL_STEPS)

    args_kwargs = dict(
        output_dir=str(TRAIN_OUT),
        overwrite_output_dir=False,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=GRAD_ACC_STEPS,
        learning_rate=5e-4,
        num_train_epochs=3.0,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        logging_steps=500,
        save_steps=SAVE_EVAL_STEPS,
        eval_steps=SAVE_EVAL_STEPS,
        save_strategy="steps",
        save_total_limit=3,
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        remove_unused_columns=False,
        report_to=[],
        seed=SEED,
        data_seed=SEED,
        optim="adamw_torch",
        max_grad_norm=1.0,
        gradient_checkpointing=True,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        dataloader_num_workers=0,
    )

    sig = inspect.signature(TrainingArguments.__init__)
    if "gradient_checkpointing_kwargs" in sig.parameters:
        args_kwargs["gradient_checkpointing_kwargs"] = {"use_reentrant": False}

    if "eval_strategy" in sig.parameters:
        args_kwargs["eval_strategy"] = "steps"
    else:
        args_kwargs["evaluation_strategy"] = "steps"

    training_args = TrainingArguments(**args_kwargs)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=RawCODeDataset(train_items),
        eval_dataset=RawCODeDataset(val_items),
        data_collator=collator,
    )

    def checkpoint_number(path):
        m = re.search(r"checkpoint-(\d+)$", path.name)
        return int(m.group(1)) if m else -1

    checkpoints = (
        sorted(TRAIN_OUT.glob("checkpoint-*"), key=checkpoint_number)
        if TRAIN_OUT.exists()
        else []
    )
    resume = str(checkpoints[-1]) if checkpoints else None
    print("Resume checkpoint:", resume)

    result = trainer.train(resume_from_checkpoint=resume)

    trainer.save_model(str(TRAIN_OUT / "best_or_final_adapter"))
    processor.save_pretrained(str(TRAIN_OUT / "processor"))
    trainer.save_state()

    train_record = {
        "metrics": result.metrics,
        "best_model_checkpoint": trainer.state.best_model_checkpoint,
        "global_step": trainer.state.global_step,
        "train_batch_size": TRAIN_BATCH_SIZE,
        "gradient_accumulation_steps": GRAD_ACC_STEPS,
        "batch_fallback_reason": batch_fallback_reason,
        "save_eval_steps": SAVE_EVAL_STEPS,
    }
    (OUT / "train_result.json").write_text(
        json.dumps(train_record, indent=2, default=str),
        encoding="utf-8",
    )

    print("Best checkpoint:", trainer.state.best_model_checkpoint)
    print("Global step:", trainer.state.global_step)

PEFT all-linear failed: ImportError('Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported')
Falling back to all discovered Linear module leaf names.
Fallback LoRA targets: ['0', '2', 'down_proj', 'gate_proj', 'k_proj', 'o_proj', 'proj', 'q_proj', 'qkv', 'up_proj', 'v_proj']


ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

## 7. Full fine-tuned evaluation

In [ ]:
if RUN_FINETUNED_EVAL:
    if not RUN_TRAIN:
        raise RuntimeError(
            "RUN_FINETUNED_EVAL=True nhưng RUN_TRAIN=False. "
            "Nếu muốn eval adapter từ run trước, cần load adapter riêng."
        )

    if hasattr(model, "gradient_checkpointing_disable"):
        model.gradient_checkpointing_disable()

    processor.tokenizer.padding_side = "left"
    model.eval()
    model.config.use_cache = True
    torch.cuda.empty_cache()

    ft_rows, ft_metrics = run_full_generation(
        "finetuned_lora3b_3epochs",
        model,
    )

## 8. Final summary

In [ ]:
summary = {
    "released_train_samples": len(train_data),
    "released_test_samples": len(test_df),
    "train_after_95_5_split": len(train_items),
    "validation_samples": len(val_items),
    "hardware": torch.cuda.get_device_name(0),
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "pillow": importlib_metadata.version("Pillow"),
    "transformers": transformers.__version__,
    "peft": peft.__version__,
    "accelerate": accelerate.__version__,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "eval_max_new_tokens": EVAL_MAX_NEW_TOKENS,
}

if RUN_ZERO_SHOT:
    summary["zero_shot"] = zero_metrics
if RUN_FINETUNED_EVAL:
    summary["finetuned"] = ft_metrics

train_result_file = OUT / "train_result.json"
if train_result_file.exists():
    summary["training"] = json.loads(
        train_result_file.read_text(encoding="utf-8")
    )

(OUT / "summary.json").write_text(
    json.dumps(summary, indent=2, default=str),
    encoding="utf-8",
)

print(json.dumps(summary, indent=2, default=str))
print()
print("Outputs:", OUT)
print("✅ FULL pipeline finished.")

# Output cần giữ

Trong `/kaggle/working/code_full_reproduction/`:

- `zero_shot_base3b_predictions.csv`
- `zero_shot_base3b_metrics.json`
- `lora_3b_3epochs/`
- `train_result.json`
- `finetuned_lora3b_3epochs_predictions.csv`
- `finetuned_lora3b_3epochs_metrics.json`
- `summary.json`

## Lưu ý khi so với paper

Đừng ghi “exact reproduction” nếu kết quả khác Table 3. Nguồn public hiện có các khác biệt/thiếu hụt quan trọng:

- paper: 3 epochs; released YAML: 1 epoch;
- paper: best validation checkpoint; public source không công bố validation split;
- evaluation script tham chiếu prompt files không có trong repo;
- paper final nói 600 test records, trong khi preprocessing notebook public chứa một intermediate split 467 trước augmentation/export.

Notebook này ghi rõ những khác biệt đó để bạn có thể báo cáo trung thực trong khóa luận.